# Analysis of US Tech Stock Returns and Risks
## 1. Introduction
This project conducts a comprehensive quantitative analysis of the return and risk characteristics of six leading U.S. technology stocks (Apple, Microsoft, Alphabet, Amazon, Tesla, Meta) from January 2021 to April 2025. The analysis covers core investment metrics including annualized return, annualized volatility, Sharpe ratio, maximum drawdown, and return correlation, providing data-driven investment insights.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot settings
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. Data Setup

In [ ]:
# Define time range and stock tickers
start_date = '2021-01-01'
end_date = '2025-04-01'

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META']
ticker_names = {
    'AAPL': 'Apple',
    'MSFT': 'Microsoft',
    'GOOGL': 'Alphabet',
    'AMZN': 'Amazon',
    'TSLA': 'Tesla',
    'META': 'Meta'
}

## 3. Data Collection & Preprocessing

In [ ]:
# Download adjusted close prices (fixed for new yfinance format)
print("Fetching historical stock data...")
data = yf.download(tickers, start=start_date, end=end_date)

# Fix for MultiIndex columns: extract Adj Close correctly
if isinstance(data.columns, pd.MultiIndex):
    data = data['Adj Close']

# Data cleaning: fill missing values and drop empty rows
data = data.ffill().bfill().dropna(how='all')
daily_returns = data.pct_change().dropna()

print(f"\nData shape after cleaning: {data.shape}")

## 4. Risk & Return Performance Calculation

In [ ]:
# Calculate core performance metrics
annual_returns = daily_returns.mean() * 252 * 100  # Annualized return (%)
annual_volatility = daily_returns.std() * np.sqrt(252) * 100  # Annualized volatility (%)

risk_free_rate = 0.02  # Assume 2% risk-free rate (U.S. Treasury yield)
sharpe_ratio = (annual_returns/100 - risk_free_rate) / (annual_volatility/100)  # Sharpe Ratio

# Calculate maximum drawdown
max_drawdown = {}
for col in daily_returns.columns:
    cum_ret = (1 + daily_returns[col]).cumprod()
    run_max = cum_ret.cummax()
    dd = (cum_ret - run_max) / run_max
    max_drawdown[col] = dd.min() * 100

# Create performance summary table
performance = pd.DataFrame({
    'Annual Return (%)': annual_returns,
    'Annual Volatility (%)': annual_volatility,
    'Sharpe Ratio': sharpe_ratio,
    'Max Drawdown (%)': max_drawdown
}).round(2)
performance['Company'] = performance.index.map(ticker_names)

print("\n===== Stock Performance Summary =====")
print(performance)

## 5. Cumulative Returns Visualization

In [ ]:
# Calculate cumulative returns
cum_returns = (1 + daily_returns).cumprod()

# Plot cumulative returns
plt.figure(figsize=(12, 6))
for col in cum_returns.columns:
    plt.plot(cum_returns.index, cum_returns[col], linewidth=2, label=ticker_names[col])
plt.title('Cumulative Returns of US Tech Stocks (2021-2025)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return (Initial $1)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Risk-Return Scatter Plot

In [ ]:
# Plot risk-return profile
plt.figure(figsize=(10, 6))
for t in tickers:
    plt.scatter(annual_volatility[t], annual_returns[t], s=150, alpha=0.8)
    plt.text(annual_volatility[t]+0.5, annual_returns[t]+0.5, ticker_names[t], fontsize=10)
plt.title('Risk-Return Profile of US Tech Stocks', fontsize=14)
plt.xlabel('Annualized Volatility (%)', fontsize=12)
plt.ylabel('Annualized Return (%)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Return Correlation Heatmap

In [ ]:
# Calculate return correlation matrix
corr = daily_returns.corr()

# Plot correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            xticklabels=[ticker_names[t] for t in corr.columns],
            yticklabels=[ticker_names[t] for t in corr.columns])
plt.title('Return Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Maximum Drawdown Visualization

In [ ]:
# Calculate drawdown series
drawdown = (cum_returns - cum_returns.cummax()) / cum_returns.cummax()

# Plot maximum drawdown
plt.figure(figsize=(12, 6))
for col in drawdown.columns:
    plt.plot(drawdown.index, drawdown[col], linewidth=1.5, label=ticker_names[col])
plt.title('Maximum Drawdown of US Tech Stocks', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Drawdown', fontsize=12)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Conclusion & Investment Insights

In [ ]:
print("\n===== Investment Insights =====")
print(f"1. Best Risk-Adjusted Return (Highest Sharpe Ratio): {performance.sort_values('Sharpe Ratio', ascending=False).iloc[0]['Company']}")
print(f"2. Highest Annual Return: {performance.sort_values('Annual Return (%)', ascending=False).iloc[0]['Company']}")
print(f"3. Lowest Volatility (Lowest Risk): {performance.sort_values('Annual Volatility (%)').iloc[0]['Company']}")
print(f"4. Smallest Maximum Drawdown (Lowest Downside Risk): {performance.sort_values('Max Drawdown (%)', ascending=False).iloc[0]['Company']}")